# Functional annotation and BERTopic analysis of discriminative *Salmonella* genes

This notebook takes the discriminative wgMLST loci, links them to EnteroBase annotations, assigns each gene to the virulent- or attenuated-enriched group, generates PubMedBERT embeddings from the annotation text, and clusters the embeddings with BERTopic.

The workflow is intended to generate the functional clusters shown in the manuscript's functional annotation analysis.


## Expected inputs

Place the following files in `data/processed/`:

| File | Description |
|---|---|
| `annotation_enterobase.tsv` | EnteroBase wgMLST annotation table. Must include columns similar to `Locus Tag` and `Description`. |
| `assigned_genes.pkl` or `assigned_genes.tsv` | Table assigning discriminative genes to `virulent` or `attenuated`. Must include a gene/locus column and a group-assignment column. |
| `virulence_conf.fa` | FASTA file containing the 1,703 discriminative gene sequences. Used only to recover the gene IDs if needed. |

Outputs are written to:

```text
results/bertopic/tables/
results/bertopic/figures/
```


In [ ]:
# Core imports
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project folders
DATA_DIR = Path("data/processed")
RESULTS_DIR = Path("results/bertopic")
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

for folder in [TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Input files
ANNOTATION_FILE = DATA_DIR / "annotation_enterobase.tsv"
ASSIGNED_GENES_FILE = DATA_DIR / "assigned_genes.pkl"  # can also be .tsv or .csv
FASTA_FILE = DATA_DIR / "virulence_conf.fa"

RANDOM_STATE = 42


In [ ]:
def check_required_files(paths):
    """Stop early with a clear message if required input files are missing."""
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required input file(s):\n"
            + "\n".join(f"  - {m}" for m in missing)
            + "\n\nUpdate DATA_DIR or place the files in data/processed/."
        )

check_required_files([ANNOTATION_FILE, ASSIGNED_GENES_FILE])


## 1. Load EnteroBase annotations and gene-group assignments

The analysis uses textual functional descriptions from EnteroBase.  
The group assignment should indicate whether each discriminative gene is enriched in virulent or attenuated serovars.


In [ ]:
def standardize_locus_id(value):
    """Standardize locus identifiers for merging across FASTA, annotation, and assignment tables."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.split()[0]        # remove trailing annotation text after spaces
    value = re.sub(r"_1$", "", value)  # remove common FASTA suffix if present
    return value

def find_column(df, candidates):
    """Return the first column matching one of several expected names."""
    lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    raise ValueError(f"None of these columns were found: {candidates}. Available columns: {list(df.columns)}")

def load_table(path):
    """Load pkl, tsv, or csv table."""
    path = Path(path)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    if path.suffix in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t")
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

annotations = pd.read_csv(ANNOTATION_FILE, sep="\t")
assigned = load_table(ASSIGNED_GENES_FILE)

locus_col = find_column(annotations, ["Locus Tag", "Locus", "gene", "Gene", "ID"])
desc_col = find_column(annotations, ["Description", "Product", "Function", "Annotation"])

assigned_gene_col = find_column(assigned, ["gene", "Gene", "Locus Tag", "Locus", "ID"])
assigned_group_col = find_column(assigned, ["group_assignment", "Group", "group", "assignment", "Set"])

annotations = annotations.copy()
assigned = assigned.copy()

annotations["gene_id"] = annotations[locus_col].apply(standardize_locus_id)
assigned["gene_id"] = assigned[assigned_gene_col].apply(standardize_locus_id)
assigned["group_assignment"] = assigned[assigned_group_col].astype(str).str.lower()

print("Annotations:", annotations.shape)
print("Assigned genes:", assigned.shape)
print(assigned["group_assignment"].value_counts(dropna=False))


In [ ]:
# Keep only annotated discriminative genes and add virulent/attenuated assignment.
df = annotations.merge(
    assigned[["gene_id", "group_assignment"]].drop_duplicates(),
    on="gene_id",
    how="inner"
)

df = df.dropna(subset=[desc_col]).copy()
df["Description"] = df[desc_col].astype(str)

# Remove exact duplicate locus IDs if present.
df = df.drop_duplicates(subset=["gene_id"])

# Keep only the two biological groups used downstream.
df = df[df["group_assignment"].isin(["virulent", "attenuated"])].copy()

df.to_csv(TABLE_DIR / "discriminative_genes_with_enterobase_annotations.tsv", sep="\t", index=False)

print("Annotated discriminative genes:", df.shape)
print(df["group_assignment"].value_counts())
df.head()


## 2. Generate PubMedBERT embeddings

Each EnteroBase annotation is converted into a sentence embedding using PubMedBERT.  
The `[CLS]` token representation is used as the annotation-level embedding.

This section may take several minutes on CPU. GPU is recommended but not required.


In [ ]:
# Install only if running in Google Colab or a fresh environment.
# Uncomment if needed:
# !pip install "transformers==4.41.2" torch tqdm

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()


In [ ]:
def embed_texts(texts, tokenizer, model, max_length=128, batch_size=32, device="cpu"):
    """Generate PubMedBERT CLS-token embeddings for a list of annotation texts."""
    embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Generating PubMedBERT embeddings"):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token embedding: one vector per annotation.
        batch_embeddings = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
        embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

descriptions = df["Description"].fillna("unknown").tolist()
embeddings = embed_texts(
    descriptions,
    tokenizer=tokenizer,
    model=bert_model,
    max_length=128,
    batch_size=32,
    device=device
)

df["embedding"] = list(embeddings)
df.to_pickle(TABLE_DIR / "discriminative_genes_pubmedbert_embeddings.pkl")

print("Embedding matrix:", embeddings.shape)


## 3. Run BERTopic separately for virulent- and attenuated-enriched genes

Separate topic models are fitted for the two groups, matching the manuscript workflow.


In [ ]:
# Install only if needed:
# !pip install bertopic==0.16.0 umap-learn hdbscan "sentence-transformers<3.0"

import umap
import hdbscan
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

def run_bertopic_for_group(data, group_name, min_cluster_size=10):
    """Run BERTopic on one gene group using precomputed PubMedBERT embeddings."""
    subset = data[data["group_assignment"] == group_name].copy()
    if subset.empty:
        raise ValueError(f"No genes found for group: {group_name}")

    texts = subset["Description"].fillna("unknown").tolist()
    X = np.vstack(subset["embedding"].values)

    # UMAP reduces the high-dimensional PubMedBERT space before HDBSCAN clustering.
    umap_model = umap.UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE
    )

    # HDBSCAN assigns genes to dense semantic clusters and leaves ambiguous genes as noise (-1).
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1
    )

    topic_model = BERTopic(
        embedding_model=None,          # embeddings are supplied directly
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = topic_model.fit_transform(texts, embeddings=X)
    subset["bertopic_cluster"] = topics

    topic_info = topic_model.get_topic_info()

    subset.to_pickle(TABLE_DIR / f"{group_name}_genes_bertopic_assignments.pkl")
    subset.to_csv(TABLE_DIR / f"{group_name}_genes_bertopic_assignments.tsv", sep="\t", index=False)
    topic_info.to_csv(TABLE_DIR / f"{group_name}_bertopic_topic_info.tsv", sep="\t", index=False)

    print(f"\n{group_name}:")
    print("  genes:", subset.shape[0])
    print("  non-noise genes:", (subset["bertopic_cluster"] != -1).sum())
    print("  clusters excluding noise:", topic_info[topic_info["Topic"] != -1].shape[0])

    return subset, topic_info, topic_model

virulent_topics, virulent_topic_info, virulent_model = run_bertopic_for_group(df, "virulent")
attenuated_topics, attenuated_topic_info, attenuated_model = run_bertopic_for_group(df, "attenuated")


## 4. Summarize BERTopic clusters

The summary tables report the number of genes per topic and the representative BERTopic keywords.


In [ ]:
def summarize_topic_table(topic_info, group_name):
    summary = topic_info.copy()
    summary = summary.rename(columns={"Topic": "bertopic_cluster", "Count": "n_genes", "Name": "topic_name"})
    summary["group_assignment"] = group_name
    return summary

topic_summary = pd.concat(
    [
        summarize_topic_table(virulent_topic_info, "virulent"),
        summarize_topic_table(attenuated_topic_info, "attenuated")
    ],
    ignore_index=True
)

topic_summary.to_csv(TABLE_DIR / "bertopic_topic_summary.tsv", sep="\t", index=False)
topic_summary.head()


## 5. Plot t-SNE visualization of PubMedBERT embeddings

This generates figure-style panels similar to the functional annotation visualization: one panel for virulent-enriched genes and one for attenuated-enriched genes.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def plot_tsne_topics(group_df, group_name, output_file):
    X = np.vstack(group_df["embedding"].values)

    tsne = TSNE(
        n_components=2,
        perplexity=min(30, max(5, (len(group_df) - 1) // 3)),
        learning_rate=200,
        init="pca",
        random_state=RANDOM_STATE
    )
    coords = tsne.fit_transform(X)

    plot_df = group_df.copy()
    plot_df["tSNE_1"] = coords[:, 0]
    plot_df["tSNE_2"] = coords[:, 1]

    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        plot_df["tSNE_1"],
        plot_df["tSNE_2"],
        c=plot_df["bertopic_cluster"],
        s=20,
        alpha=0.85
    )
    plt.title(f"PubMedBERT embeddings of {group_name} genes")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.colorbar(scatter, label="BERTopic cluster")
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.show()

    return plot_df

virulent_plot = plot_tsne_topics(
    virulent_topics,
    "virulent-enriched",
    FIGURE_DIR / "pubmedbert_bertopic_virulent_tsne.png"
)

attenuated_plot = plot_tsne_topics(
    attenuated_topics,
    "attenuated-enriched",
    FIGURE_DIR / "pubmedbert_bertopic_attenuated_tsne.png"
)

virulent_plot.to_csv(TABLE_DIR / "virulent_bertopic_tsne_coordinates.tsv", sep="\t", index=False)
attenuated_plot.to_csv(TABLE_DIR / "attenuated_bertopic_tsne_coordinates.tsv", sep="\t", index=False)


## 6. Optional: inspect clusters of biological interest

Use these cells after running the notebook to inspect specific topics, for example clusters enriched for fimbriae, SPI genes, phage genes, or metal-resistance genes.


In [ ]:
def search_descriptions(data, pattern, group=None):
    """Search annotation descriptions for biologically relevant keywords."""
    subset = data.copy()
    if group is not None:
        subset = subset[subset["group_assignment"] == group]
    mask = subset["Description"].str.contains(pattern, case=False, na=False, regex=True)
    cols = ["gene_id", "group_assignment", "bertopic_cluster", "Description"]
    return subset.loc[mask, cols].sort_values(["group_assignment", "bertopic_cluster", "gene_id"])

# Examples:
spi_genes = search_descriptions(pd.concat([virulent_topics, attenuated_topics]), r"SPI|type III|invasion|secretion")
fimbrial_genes = search_descriptions(pd.concat([virulent_topics, attenuated_topics]), r"fimbria|fimbrial|pilus")
metal_resistance_genes = search_descriptions(pd.concat([virulent_topics, attenuated_topics]), r"arsenic|arsenate|arsenical|mercury|copper")

spi_genes.head()
